# Pre-processing : création d'un dataframe regroupant l'ancienne et la nouvelle base de donnée.

Voici les étapes principales : 
1. Importer les données
2. Créer un dataframe pour 2022-2024 regroupant les donneés "Lieux", "Usagers", "Caractéristiques"
3. Idem pour 2005-2021
4. Concaténer les deux dataframes
5. One-hot encoding et abandonner les variables qui ne sont pas pertinentes



## 1. Importer les données

In [650]:
import pandas as pd
from IPython.display import display
import polars as pl

In [651]:
# Téléchargement des dataframes depuis la base de données des BAAC.
urls_y = {
    2022: {
        "carac" : "https://www.data.gouv.fr/api/1/datasets/r/5fc299c0-4598-4c29-b74c-6a67b0cc27e7",
        "lieux" : "https://www.data.gouv.fr/api/1/datasets/r/a6ef711a-1f03-44cb-921a-0ce8ec975995",
        "usagers" : "https://www.data.gouv.fr/api/1/datasets/r/62c20524-d442-46f5-bfd8-982c59763ec8"
    },
    2023 : {
        "carac" : "https://www.data.gouv.fr/api/1/datasets/r/104dbb32-704f-4e99-a71e-43563cb604f2",
        "lieux" : "https://www.data.gouv.fr/api/1/datasets/r/8bef19bf-a5e4-46b3-b5f9-a145da4686bc",
        "usagers" : "https://www.data.gouv.fr/api/1/datasets/r/68848e2a-28dd-4efc-9d5f-d512f7dbe66f"
    },
    2024 : {
        "carac" : "https://www.data.gouv.fr/api/1/datasets/r/83f0fb0e-e0ef-47fe-93dd-9aaee851674a",
        "lieux" : "https://www.data.gouv.fr/api/1/datasets/r/228b3cda-fdfb-4677-bd54-ab2107028d2d",
        "usagers" : "https://www.data.gouv.fr/api/1/datasets/r/f57b1f58-386d-4048-8f78-2ebe435df868"
    }
}

url_05_21 = {
    "carac" : "https://www.data.gouv.fr/api/1/datasets/r/a3cac8bc-4a07-4124-8a08-633a3a91d40b",
    "lieux" : "https://www.data.gouv.fr/api/1/datasets/r/b7f25e45-de32-4801-b0eb-62989f1a7406",
    "usagers" : "https://www.data.gouv.fr/api/1/datasets/r/a64b1b9f-4d56-4b26-ae90-9f40b878e109"
}

# Dictionnaire qui contiendra les dataframes pour chaque année en vue de la concaténation

dfs = {}

yrl = [2022,2023,2024]
names = ["carac","lieux","usagers"]

for name in names:
    list_df = []
    for yr in yrl:
        url = urls_y[yr][name]
        df_yr = pd.read_csv(url,sep=";") # Le séparateur utilisé est ";"
        df_yr.insert(1,"year",yr) # Colonne année pour distinguer les accidents entre années en vue des opérations de fusion de dataframes
        list_df.append(df_yr)

    concatenated_df_type = pd.concat(list_df,ignore_index=True) # On fait fi de l'index
    dfs[name] = concatenated_df_type # On associe le dataframe des trois années au type de dataframe

dfs

# Prend environ 40 secondes à tourner

C:\Users\Lucy Neveux\AppData\Local\Temp\ipykernel_113376\4211296108.py:37: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_yr = pd.read_csv(url,sep=";") # Le séparateur utilisé est ";"
C:\Users\Lucy Neveux\AppData\Local\Temp\ipykernel_113376\4211296108.py:37: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df_yr = pd.read_csv(url,sep=";") # Le séparateur utilisé est ";"


{'carac':          Accident_Id  year  jour  mois    an   hrmn  lum dep    com  agg  int  \
 0       2.022000e+11  2022    19    10  2022  16:15    1  26  26198    2    3   
 1       2.022000e+11  2022    20    10  2022  08:34    1  25  25204    2    3   
 2       2.022000e+11  2022    20    10  2022  17:15    1  22  22360    2    6   
 3       2.022000e+11  2022    20    10  2022  18:00    1  16  16102    2    3   
 4       2.022000e+11  2022    19    10  2022  11:45    1  13  13103    1    1   
 ...              ...   ...   ...   ...   ...    ...  ...  ..    ...  ...  ...   
 164521           NaN  2024    10     7  2024  02:05    5  94  94065    2    3   
 164522           NaN  2024    30    11  2024  15:27    1  92  92062    2    1   
 164523           NaN  2024    24    10  2024  20:40    3  29  29068    1    1   
 164524           NaN  2024    30    11  2024  15:40    1  92  92012    2    1   
 164525           NaN  2024    10     7  2024  08:30    1  94  94028    2    1   
 
     

## 2. Créer un dataframe pour 2022-2024 regroupant les donneés "Lieux", "Usagers", "Caractéristiques"

In [652]:
if "an" in dfs["carac"].columns:
    dfs["carac"].drop(columns=["an"],inplace=True) # Colonne an redondante avec colonne year

In [653]:
if "Accident_Id" in dfs["carac"].columns:
    dfs["carac"]["Num_Acc"] = dfs["carac"]["Num_Acc"].fillna(dfs["carac"]["Accident_Id"])   # On fusionne Accident_Id et Num_Acc, qui représentent le même indicateur
    dfs["carac"].drop(columns=["Accident_Id"],inplace=True)

col_data = dfs["carac"].pop("Num_Acc")
dfs["carac"].insert(0,"Num_Acc",col_data) # On replace la colonne "Num_Acc" en index 0 des colonnes du dataframe

#### Analyses préliminaires

In [654]:
for n,content in dfs.items(): 
    print(n,len(content))

carac 164526
lieux 196410
usagers 377638


In [655]:
sum(dfs["carac"]["Num_Acc"].value_counts()>1)

0

Dans "carac" il n'y a bien qu'une ligne par accident. 
Il est normal que le dataframe "usagers" soit le plus long : en effet, pour chaque accident corporel, on peut compter plusieurs victimes. Il est plus surprenant que "lieux" soit plus long que "carac". 
Observons les doublons :

In [656]:
dfs["lieux"]["Num_Acc"].value_counts()

Num_Acc
202300035508    5
202400050330    5
202300039940    4
202300034748    4
202300006227    4
               ..
202200000042    1
202200000043    1
202200000044    1
202200000045    1
202200000006    1
Name: count, Length: 164526, dtype: int64

In [657]:
dfs["lieux"][dfs["lieux"]["Num_Acc"]==202300035508]

,Num_Acc,year,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma
101231,202300035508,2023,4,BOULEVARD DE BEAUSEJOUR,0,NaN,1,0,0,1,0,0,1,NaN,-1,1,0,1,30
101232,202300035508,2023,4,BOULEVARD EMILE AUGIER,0,NaN,3,2,0,1,-1,-1,1,NaN,-1,1,9,1,50
101233,202300035508,2023,4,CHAUSSEE LA MUETTE,0,NaN,3,1,3,1,-1,-1,1,NaN,-1,1,6,1,50
101234,202300035508,2023,4,RUE D ANDIGNE,0,NaN,1,1,0,1,0,0,1,NaN,-1,1,0,1,30
101235,202300035508,2023,4,RUE LARGILLIERE,0,NaN,2,2,0,1,0,0,1,NaN,-1,1,0,1,30


"Lieux" indique les différentes voies liées à l'accident lorsqu'il a eu lieu dans une intersection complexe. Il contient par ailleurs différentes informations sur le lieu de l'accident (type de route **catr**, l'état de la surface **surf**, la vitesse maximale autorisée **vma**).

Les données les plus générales sont les "caractéristiques de l'accident", uniques pour chaque accident. Dans chaque accident, on a des données sur le lieu de l'accident et l'avant-accident qui est différent selon les acteurs impliqués. Enfin, la partie "usagers" comporte des informations sur chacun des usagers impliqués dans l'accident ; c'est le dataset le plus large.

#### Observons "Lieux"

Nous supprimons la colonne voie, qui est bruitée, et peu informative. On supprime donc également v1 et v2, qui donnent l'adresse. On supprime de même les colonnes concernant les bornes kilométriques (pr et pr1)

A contrario, la VMA (vitesse maximale autorisée), la catégorie de route (catr - autoroute, route urbaine...), l'état de la surface (surf - conditions de la route : verglas, pluie, neige), le régime de circulation (circ - bidirectionnel, unidirectionnel,...), le type d'infrastructure (infra - ponts, tunnels ou carrefours), situation de l'accident (situ - où a précisément eu lieu l'accident : sur la chaussée, sur la bande d'arrêt d'urgence,...), sont particulièrement importantes pour notre prédiction.

Les colonnes plan, prof, nbv, lartpc, larrout, vosp, donnent des informations précises sur la configuration des lieux de l'accident. En particulier, prof (topographie), plan (courbure de la route), larrout (largeur de la route) sont très intéressantes.

In [658]:
dfs["lieux"].drop(columns=["voie","v1","v2", "pr", "pr1"], inplace=True)
dfs["lieux"]

,Num_Acc,year,catr,circ,nbv,vosp,prof,plan,lartpc,larrout,surf,infra,situ,vma
0,202200000001,2022,4,2,2,0,1,1,NaN,-1,1,0,1,50
1,202200000002,2022,4,2,2,0,1,1,NaN,-1,1,0,1,50
2,202200000003,2022,3,-1,2,0,1,1,NaN,-1,1,5,1,50
3,202200000004,2022,4,1,1,0,2,1,NaN,4,1,0,1,30
4,202200000005,2022,3,2,2,0,1,1,NaN,-1,1,0,1,80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196405,202400054398,2024,3,1,1,-1,1,1,NaN,-1,2,5,4,-1
196406,202400054399,2024,3,1,1,2,1,2,NaN,-1,1,0,1,30
196407,202400054400,2024,2,1,2,0,2,1,NaN,10,1,0,1,110
196408,202400054401,2024,3,2,3,0,1,1,NaN,"10,5",1,0,1,50


In [659]:
dfs["lieux"][dfs["lieux"]["Num_Acc"]==202300000001]

,Num_Acc,year,catr,circ,nbv,vosp,prof,plan,lartpc,larrout,surf,infra,situ,vma
55302,202300000001,2023,4,1,2,0,1,1,NaN,-1,2,0,1,30
55303,202300000001,2023,4,1,1,0,1,1,NaN,-1,2,0,1,30


Retirons maintenant les doublons.

In [660]:
# Garde uniquement la première occurrence de chaque Num_Acc
dfs["lieux"] = dfs["lieux"].drop_duplicates(subset=["Num_Acc"], keep='first')

In [661]:
dfs["lieux"][dfs["lieux"]["Num_Acc"]==202300000001]

,Num_Acc,year,catr,circ,nbv,vosp,prof,plan,lartpc,larrout,surf,infra,situ,vma
55302,202300000001,2023,4,1,2,0,1,1,NaN,-1,2,0,1,30


#### Observons "Usagers"

Nous supprimons la colonne id_usager ainsi que id_véhicule et num_véhicule, car nous ne nous attarderons pas sur l'analyse des véhicules et nous avons déjà Num_Acc pour identifier les accidents.

In [662]:
dfs["usagers"].drop(columns=["id_usager","id_vehicule","num_veh"], inplace=True)
dfs["usagers"]

,Num_Acc,year,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
0,202200000001,2022,1,1,3,1,2008.0,5,2,8,-1,-1,-1,-1
1,202200000001,2022,1,1,1,1,1948.0,5,1,8,-1,-1,-1,-1
2,202200000002,2022,1,1,4,1,1988.0,9,1,0,-1,0,0,-1
3,202200000002,2022,1,1,1,1,1970.0,4,1,0,-1,0,0,-1
4,202200000003,2022,1,1,1,1,2002.0,0,1,0,-1,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377633,202400054401,2024,1,1,4,2,1978.0,0,0,0,0,-1,-1,-1
377634,202400054401,2024,1,1,1,1,1984.0,0,2,6,0,-1,-1,-1
377635,202400054402,2024,1,1,4,1,1981.0,4,1,0,-1,-1,-1,-1
377636,202400054402,2024,1,1,4,2,1986.0,9,1,0,-1,-1,-1,-1


#### Observons "Caractéristiques"

Nous pouvons enlever la colonne "adr" car nous avons déjà la latitude et la longitude. Pour la même raison, nous pouvons retirer "dep" et "com", mais nous les gardons de coté si jamais nous souhaitons faire une analyse en fonction des départements et des communes plus tard.

In [663]:
df_geo_backup_2224 = dfs["carac"][["Num_Acc", "year", "dep", "com"]].copy()
dfs["carac"].drop(columns=["dep","com","adr"], inplace=True) 
dfs["carac"]

,Num_Acc,year,jour,mois,hrmn,lum,agg,int,atm,col,lat,long
0,2.022000e+11,2022,19,10,16:15,1,2,3,1,3,"44,5594200000","4,7257200000"
1,2.022000e+11,2022,20,10,08:34,1,2,3,1,3,"46,9258100000","6,3462000000"
2,2.022000e+11,2022,20,10,17:15,1,2,6,1,2,"48,4931620000","-2,7604390000"
3,2.022000e+11,2022,20,10,18:00,1,2,3,8,6,"45,6926520000","-0,3262900000"
4,2.022000e+11,2022,19,10,11:45,1,1,1,1,2,"43,6755790366","5,0927031775"
...,...,...,...,...,...,...,...,...,...,...,...,...
164521,2.024001e+11,2024,10,7,02:05,5,2,3,2,6,"48,75740000","2,34469000"
164522,2.024001e+11,2024,30,11,15:27,1,2,1,1,6,"48,88694322","2,24863619"
164523,2.024001e+11,2024,24,10,20:40,3,1,1,1,1,"48,52619024","-3,96317650"
164524,2.024001e+11,2024,30,11,15:40,1,2,1,1,3,"48,83286197","2,24394009"


#### Fusionner 2022, 2023 et 2024

Créons maintenant un premier dataset, qui fusionne les données de 2022,2023, et 2024 pour les dataframes **usagers** et **carac** :

In [664]:
KEY = ["year","Num_Acc"]

df_final2224 = dfs["usagers"].merge(dfs["carac"],on=KEY,how="left") # On garde toutes les colonnes usagers

print(df_final2224.shape[0]==dfs["usagers"].shape[0])

# df_final2224 et dfs["usagers"] font bien la même longueur.

True


Ajoutons-y les données sur les lieux. 

In [665]:
df_final2224 = df_final2224.merge(dfs["lieux"],on=KEY,how="left") 


In [666]:
df_final2224

,Num_Acc,year,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,...,nbv,vosp,prof,plan,lartpc,larrout,surf,infra,situ,vma
0,202200000001,2022,1,1,3,1,2008.0,5,2,8,...,2,0,1,1,NaN,-1,1,0,1,50
1,202200000001,2022,1,1,1,1,1948.0,5,1,8,...,2,0,1,1,NaN,-1,1,0,1,50
2,202200000002,2022,1,1,4,1,1988.0,9,1,0,...,2,0,1,1,NaN,-1,1,0,1,50
3,202200000002,2022,1,1,1,1,1970.0,4,1,0,...,2,0,1,1,NaN,-1,1,0,1,50
4,202200000003,2022,1,1,1,1,2002.0,0,1,0,...,2,0,1,1,NaN,-1,1,5,1,50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377633,202400054401,2024,1,1,4,2,1978.0,0,0,0,...,3,0,1,1,NaN,"10,5",1,0,1,50
377634,202400054401,2024,1,1,1,1,1984.0,0,2,6,...,3,0,1,1,NaN,"10,5",1,0,1,50
377635,202400054402,2024,1,1,4,1,1981.0,4,1,0,...,2,0,2,1,NaN,-1,1,2,1,50
377636,202400054402,2024,1,1,4,2,1986.0,9,1,0,...,2,0,2,1,NaN,-1,1,2,1,50


In [667]:
sum(df_final2224["grav"].isna())

0

In [668]:
df_final2224

,Num_Acc,year,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,...,nbv,vosp,prof,plan,lartpc,larrout,surf,infra,situ,vma
0,202200000001,2022,1,1,3,1,2008.0,5,2,8,...,2,0,1,1,NaN,-1,1,0,1,50
1,202200000001,2022,1,1,1,1,1948.0,5,1,8,...,2,0,1,1,NaN,-1,1,0,1,50
2,202200000002,2022,1,1,4,1,1988.0,9,1,0,...,2,0,1,1,NaN,-1,1,0,1,50
3,202200000002,2022,1,1,1,1,1970.0,4,1,0,...,2,0,1,1,NaN,-1,1,0,1,50
4,202200000003,2022,1,1,1,1,2002.0,0,1,0,...,2,0,1,1,NaN,-1,1,5,1,50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377633,202400054401,2024,1,1,4,2,1978.0,0,0,0,...,3,0,1,1,NaN,"10,5",1,0,1,50
377634,202400054401,2024,1,1,1,1,1984.0,0,2,6,...,3,0,1,1,NaN,"10,5",1,0,1,50
377635,202400054402,2024,1,1,4,1,1981.0,4,1,0,...,2,0,2,1,NaN,-1,1,2,1,50
377636,202400054402,2024,1,1,4,2,1986.0,9,1,0,...,2,0,2,1,NaN,-1,1,2,1,50


La gravité des blessures de chaque victime est renseignée, ce qui nous évite un travail supplémentaire de preprocessing pour la variable d'intérêt **grav**.

>Pour rappel, **grav** peut prendre quatre valeurs : 
>* 1 = Indemne
>* 2 = Tué
>* 3 = Blessé hospitalisé
>* 4 = Blessé léger
    

## 3. Créer un dataframe pour 2005-2021 regroupant les donneés "Lieux", "Usagers", "Caractéristiques"

#### Créer df0521

In [669]:
url_05_21.items()

dict_items([('carac', 'https://www.data.gouv.fr/api/1/datasets/r/a3cac8bc-4a07-4124-8a08-633a3a91d40b'), ('lieux', 'https://www.data.gouv.fr/api/1/datasets/r/b7f25e45-de32-4801-b0eb-62989f1a7406'), ('usagers', 'https://www.data.gouv.fr/api/1/datasets/r/a64b1b9f-4d56-4b26-ae90-9f40b878e109')])

Nous construisons maintenant le dataset contenant les données de 2005 à 2021 en vue d'une harmonisation

In [670]:
df0521={}
for type,link in url_05_21.items():
    df0521[type] = pd.read_csv(link,
                               encoding="latin-1",  # Old files, with a different encoding than the recent ones
                               sep=",")
df0521

# Prend environ 2 min 30 à tourner

C:\Users\Lucy Neveux\AppData\Local\Temp\ipykernel_113376\3640247353.py:3: DtypeWarning: Columns (4,10,13,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df0521[type] = pd.read_csv(link,
C:\Users\Lucy Neveux\AppData\Local\Temp\ipykernel_113376\3640247353.py:3: DtypeWarning: Columns (3,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df0521[type] = pd.read_csv(link,
C:\Users\Lucy Neveux\AppData\Local\Temp\ipykernel_113376\3640247353.py:3: DtypeWarning: Columns (9,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df0521[type] = pd.read_csv(link,


{'carac':          Unnamed: 0       num_acc  mois  jour   hrmn  lum  agg  int  atm  col  \
 0                 1  200500000001     1    12   1900    3    2    1  1.0  3.0   
 1                 2  200500000002     1    21   1600    1    2    1  1.0  1.0   
 2                 3  200500000003     1    21   1845    3    1    1  2.0  1.0   
 3                 4  200500000004     1     4   1615    1    1    1  1.0  5.0   
 4                 5  200500000005     1    10   1945    3    1    1  3.0  6.0   
 ...             ...           ...   ...   ...    ...  ...  ...  ...  ...  ...   
 1121566     1121567  202100056514     1     1  06:10    3    1    1  5.0  6.0   
 1121567     1121568  202100056515     1     1  10:20    1    1    1  2.0  6.0   
 1121568     1121569  202100056516     1     1  18:00    3    1    1  2.0  1.0   
 1121569     1121570  202100056517     1     1  10:55    1    1    2  1.0  6.0   
 1121570     1121571  202100056518     1     2  18:00    3    1    1  3.0  1.0   
 
     

#### Harmonisation préliminaire

On supprime la colonne redondante d'index "Unnamed: 0" (liée au format d'importation) et on renomme les colonnes "num_acc" et "annee" pour être en cohérence avec le dataset df_final2224 :

In [671]:
for type in ["usagers","carac"]:
    df0521[type]=df0521[type].rename(columns={"annee":"year","num_acc":"Num_Acc"})
    if "Unnamed: 0" in df0521[type].columns:
        df0521[type].drop(columns="Unnamed: 0",inplace=True)

df0521


{'carac':               Num_Acc  mois  jour   hrmn  lum  agg  int  atm  col    com  \
 0        200500000001     1    12   1900    3    2    1  1.0  3.0     11   
 1        200500000002     1    21   1600    1    2    1  1.0  1.0     51   
 2        200500000003     1    21   1845    3    1    1  2.0  1.0     51   
 3        200500000004     1     4   1615    1    1    1  1.0  5.0     82   
 4        200500000005     1    10   1945    3    1    1  3.0  6.0    478   
 ...               ...   ...   ...    ...  ...  ...  ...  ...  ...    ...   
 1121566  202100056514     1     1  06:10    3    1    1  5.0  6.0  33021   
 1121567  202100056515     1     1  10:20    1    1    1  2.0  6.0  38405   
 1121568  202100056516     1     1  18:00    3    1    1  2.0  1.0  26064   
 1121569  202100056517     1     1  10:55    1    1    2  1.0  6.0  33003   
 1121570  202100056518     1     2  18:00    3    1    1  3.0  1.0  78423   
 
                                adr gps            lat           

In [672]:
df0521["lieux"]=df0521["lieux"].rename(columns={"annee":"year","num_acc":"Num_Acc"})
if "Unnamed: 0" in df0521["lieux"].columns:
    df0521["lieux"].drop(columns="Unnamed: 0",inplace=True)

#### Supprimer les colonnes inutiles

Maintenant, nous enlevons les mêmes colonnes que nous avions enlevées dans dfs. 

In [673]:
df0521["lieux"].drop(columns=["voie","v1","v2", "pr", "pr1", "env1"], inplace=True) #en1 n'est que présente entre 2005 et 2021 et signifie la proximité avec une école
df0521["usagers"].drop(columns=["num_veh", "id_vehicule"], inplace=True)
df0521["carac"].drop(columns=["dep","com","adr", "gps"], inplace=True) #gps n'est pas présent dans les données 2022-2024, et nous pouvons la supprimer car nous avons long et lat
df0521


{'carac':               Num_Acc  mois  jour   hrmn  lum  agg  int  atm  col  \
 0        200500000001     1    12   1900    3    2    1  1.0  3.0   
 1        200500000002     1    21   1600    1    2    1  1.0  1.0   
 2        200500000003     1    21   1845    3    1    1  2.0  1.0   
 3        200500000004     1     4   1615    1    1    1  1.0  5.0   
 4        200500000005     1    10   1945    3    1    1  3.0  6.0   
 ...               ...   ...   ...    ...  ...  ...  ...  ...  ...   
 1121566  202100056514     1     1  06:10    3    1    1  5.0  6.0   
 1121567  202100056515     1     1  10:20    1    1    1  2.0  6.0   
 1121568  202100056516     1     1  18:00    3    1    1  2.0  1.0   
 1121569  202100056517     1     1  10:55    1    1    2  1.0  6.0   
 1121570  202100056518     1     2  18:00    3    1    1  3.0  1.0   
 
                    lat            long  year  
 0            5051500.0        294400.0  2005  
 1            5053700.0        280200.0  2005  
 2   

In [674]:
dfs["lieux"] = dfs["lieux"].drop_duplicates(subset=["Num_Acc"], keep='first')

#### Merge

In [675]:
KEY = ["Num_Acc","year"]
df_final0521 = df0521["usagers"].merge(df0521["carac"],on=KEY,how="left")   # On garde toutes les données de "usagers", qui est plus large que "carac".
df_final0521

,Num_Acc,place,catu,grav,sexe,trajet,secu,locp,actp,etatp,...,mois,jour,hrmn,lum,agg,int,atm,col,lat,long
0,2.005000e+11,1.0,1,4,1,1.0,11.0,0.0,0,0.0,...,1.0,12.0,1900,3.0,2.0,1.0,1.0,3.0,5051500.0,294400.0
1,2.005000e+11,1.0,1,3,2,3.0,11.0,0.0,0,0.0,...,1.0,12.0,1900,3.0,2.0,1.0,1.0,3.0,5051500.0,294400.0
2,2.005000e+11,2.0,2,1,1,0.0,11.0,0.0,0,0.0,...,1.0,12.0,1900,3.0,2.0,1.0,1.0,3.0,5051500.0,294400.0
3,2.005000e+11,4.0,2,1,1,0.0,31.0,0.0,0,0.0,...,1.0,12.0,1900,3.0,2.0,1.0,1.0,3.0,5051500.0,294400.0
4,2.005000e+11,5.0,2,1,1,0.0,11.0,0.0,0,0.0,...,1.0,12.0,1900,3.0,2.0,1.0,1.0,3.0,5051500.0,294400.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2509615,2.021001e+11,1.0,1,4,1,0.0,NaN,0.0,0,-1.0,...,1.0,1.0,18:00,3.0,1.0,1.0,2.0,1.0,"44,9112100000","5,0196360000"
2509616,2.021001e+11,1.0,1,4,1,5.0,NaN,0.0,0,-1.0,...,1.0,1.0,18:00,3.0,1.0,1.0,2.0,1.0,"44,9112100000","5,0196360000"
2509617,2.021001e+11,1.0,1,3,1,0.0,NaN,0.0,0,-1.0,...,1.0,1.0,10:55,1.0,1.0,2.0,1.0,6.0,"44,9542747363","-0,5179211363"
2509618,2.021001e+11,1.0,1,3,1,3.0,NaN,-1.0,-1,-1.0,...,1.0,2.0,18:00,3.0,1.0,1.0,3.0,1.0,"48,7966700000","2,0505000000"


Refaisons le même travail que précedemment sur Lieux.Remarquons que dans le dataset ancien figure le feature "env1" qui signifie la proximité d'une école, nous pouvons retirer ce feature. 

In [676]:
df_final0521 = df_final0521.merge(df0521["lieux"],on=KEY,how="left") 

In [677]:
df_final0521

,Num_Acc,place,catu,grav,sexe,trajet,secu,locp,actp,etatp,...,nbv,vosp,prof,plan,lartpc,larrout,surf,infra,situ,vma
0,2.005000e+11,1.0,1,4,1,1.0,11.0,0.0,0,0.0,...,2.0,0.0,1.0,1.0,0.0,63.0,1.0,0.0,1.0,NaN
1,2.005000e+11,1.0,1,3,2,3.0,11.0,0.0,0,0.0,...,2.0,0.0,1.0,1.0,0.0,63.0,1.0,0.0,1.0,NaN
2,2.005000e+11,2.0,2,1,1,0.0,11.0,0.0,0,0.0,...,2.0,0.0,1.0,1.0,0.0,63.0,1.0,0.0,1.0,NaN
3,2.005000e+11,4.0,2,1,1,0.0,31.0,0.0,0,0.0,...,2.0,0.0,1.0,1.0,0.0,63.0,1.0,0.0,1.0,NaN
4,2.005000e+11,5.0,2,1,1,0.0,11.0,0.0,0,0.0,...,2.0,0.0,1.0,1.0,0.0,63.0,1.0,0.0,1.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2509615,2.021001e+11,1.0,1,4,1,0.0,NaN,0.0,0,-1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2509616,2.021001e+11,1.0,1,4,1,5.0,NaN,0.0,0,-1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2509617,2.021001e+11,1.0,1,3,1,0.0,NaN,0.0,0,-1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2509618,2.021001e+11,1.0,1,3,1,3.0,NaN,-1.0,-1,-1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Concaténer les deux dataframes

Observons les différences dans les noms de colonnes du format des anciens datasets (2005-2021), et des nouveaux datasets (2022-2024).

In [678]:
cols_recentes = set(df_final2224.columns)
cols_anciennes = set(df_final0521.columns)

unique_recent = cols_recentes-cols_anciennes # Soustraire les sets permet de ne garder que les colonnes uniquement présentes en 2022-2024
unique_ancien = cols_anciennes-cols_recentes # Colonnes uniquement présentes en 2005-2021

print(unique_ancien,unique_recent)
print(cols_anciennes,cols_recentes)

{'secu'} set()
{'larrout', 'sexe', 'grav', 'hrmn', 'plan', 'situ', 'agg', 'secu', 'locp', 'year', 'secu3', 'atm', 'vosp', 'secu1', 'nbv', 'long', 'actp', 'int', 'etatp', 'circ', 'infra', 'jour', 'vma', 'an_nais', 'mois', 'lartpc', 'surf', 'trajet', 'col', 'secu2', 'lat', 'lum', 'catu', 'catr', 'prof', 'Num_Acc', 'place'} {'larrout', 'sexe', 'grav', 'hrmn', 'plan', 'situ', 'agg', 'locp', 'year', 'secu3', 'atm', 'vosp', 'secu1', 'nbv', 'long', 'actp', 'int', 'etatp', 'circ', 'infra', 'jour', 'vma', 'an_nais', 'mois', 'lartpc', 'surf', 'trajet', 'col', 'secu2', 'lat', 'lum', 'catu', 'catr', 'prof', 'Num_Acc', 'place'}


#### Analyse de la donnée securité (one-hot encoding)



La colonne "secu" correspond de 2005 à 2021 à un code sur deux caractères : 
- Le premier concerne l'existence d'un équipement de sécurité : 
    1 – Ceinture
    2 – Casque
    3 – Dispositif enfants
    4 – Equipement réfléchissant
    9 – Autre 

- Le second concerne l'utilisation de cet équipement de sécurité :
    1 – Oui
    2 – Non
    3 – Non déterminable

Dans les versions plus récentes, il est question de l'existence ET de l'utilisation d'un équipement de sécurité, jusqu'à trois à la fois (secu1,secu2,secu3):
    -1 – Non renseigné  
    0 – Aucun équipement  
    1 – Ceinture  
    2 – Casque  
    3 – Dispositif enfants  
    4 – Gilet réfléchissant  
    5 – Airbag (2RM/3RM)  
    6 – Gants (2RM/3RM)  
    7 – Gants + Airbag (2RM/3RM)  
    8 – Non déterminable  
    9 – Autre

Notons que les lignes de code ci-dessus suggèreent que "secu" n'est pas dans la base de données 2022-2024, mais que "secu1", "secu2", "secu3" dans dans la base de données 2005-2021. En effet, on peut supposer qu'un effort d'harmonisation des bases a été entamé. 

In [679]:
# regardons si ces colonnes sont toutes vides entre 2005 et 2021 (ce n'est visiblement pas les cas)
print(df_final0521[['secu1', 'secu2', 'secu3']].isna().all())

secu1    False
secu2    False
secu3    False
dtype: bool


In [680]:
# regardons la répartition des valeurs dans ces colonnes
print(df_final0521[['secu1', 'secu2', 'secu3']].apply(pd.Series.value_counts).head(10))

       secu1   secu2   secu3
-1.0    2912  134830  363199
 0.0   28989  146876    1102
 1.0  218344     670      21
 2.0   68650     739       7
 3.0    2384     383       5
 4.0     232    3363      51
 5.0     194    4560      19
 6.0     302   33212     190
 7.0      10     505       8
 8.0   44988   41228     164


In [681]:
# On regarde le nombre de valeurs non-nulles par année pour chaque colonne
verif_secu = df_final0521.groupby('year')[['secu', 'secu1', 'secu2', 'secu3']].count()
print(verif_secu)


        secu   secu1   secu2   secu3
year                                
2005  197498       0       0       0
2006  187084       0       0       0
2007  188457       0       0       0
2008  170960       0       0       0
2009  160953       0       0       0
2010  150001       0       0       0
2011  144045       0       0       0
2012  134397       0       0       0
2013  124695       0       0       0
2014  128151       0       0       0
2015  122244       0       0       0
2016  124062       0       0       0
2017  127071       0       0       0
2018  126040       0       0       0
2019       0  132977  132977  132977
2020       0  105295  105295  105295
2021       0  129153  129153  129153


On constate qu'il y a eu une réforme en 2019. L'harmonisation portera donc sur les données de 2005 à 2018.


Pour cela, nous proposons de supprimer les colonnes "secu1", "secu2", et "secu3" (ainsi que "secu" pour l'ancienne base de données), et à la place mettre 11 colonnes (pour "non renseigné", "aucun équipement", "ceinture", "casque", etc...), dans lequel 1 signifie que l'équiment existe et est utilisé, et 0 signifie qu'il n'existe pas ou n'a pas été utilisé. (Nous effectuons un one-hot encoding)

In [682]:
cols_final = ["secu_-1", "secu_0", "secu_1", "secu_2", "secu_3", "secu_4", "secu_5", "secu_6", "secu_7", "secu_8", "secu_9"]
secu_cols = ["secu1", "secu2", "secu3"]

Regardons le dataframe de 2022 à 2024

In [683]:
# Initialiser les colonnes finales à 0
for col in cols_final:
    df_final2224[col] = 0

# Remplir les colonnes finales
for c in secu_cols:
    dummies = pd.get_dummies(df_final2224[c], prefix='secu')
    for col in dummies.columns:
        df_final2224[col] = df_final2224[col] | dummies[col]

# Supprimer les colonnes originales
df_final2224.drop(columns=secu_cols, inplace=True)


In [684]:
print(df_final2224.columns.tolist())

['Num_Acc', 'year', 'place', 'catu', 'grav', 'sexe', 'an_nais', 'trajet', 'locp', 'actp', 'etatp', 'jour', 'mois', 'hrmn', 'lum', 'agg', 'int', 'atm', 'col', 'lat', 'long', 'catr', 'circ', 'nbv', 'vosp', 'prof', 'plan', 'lartpc', 'larrout', 'surf', 'infra', 'situ', 'vma', 'secu_-1', 'secu_0', 'secu_1', 'secu_2', 'secu_3', 'secu_4', 'secu_5', 'secu_6', 'secu_7', 'secu_8', 'secu_9']


On fait exactement la même chose sur l'ancienne base de données de 2019 à 2021. 

In [685]:

#  Initialiser les colonnes finales à 0 
for col in cols_final:
    df_final0521[col] = 0

# Remplir les colonnes finales (ne touchera que les lignes où secu1/2/3 sont remplis, donc 2019-2021)
for c in secu_cols:
    # On génère les dummies
    dummies = pd.get_dummies(df_final0521[c], prefix='secu')
    
    # On s'assure de ne fusionner que les colonnes qui existent dans cols_final
    for col in dummies.columns:
        if col in df_final0521.columns:
            df_final0521[col] = df_final0521[col] | dummies[col]

# Supprimer les colonnes originales (elles ne servent plus pour 2019-2021)
df_final0521.drop(columns=secu_cols, inplace=True)

Maintenant travaillons sur les années 2005-2018.

In [ ]:
# faire pareil pour 2005-2018

In [647]:
# Vérification sur l'année 2010 
print(df_final0521[df_final0521['year'] == 2010][cols_final].sum())

secu_-1        0
secu_0       668
secu_1     85786
secu_2     28635
secu_3      1073
secu_4       190
secu_5         0
secu_6         0
secu_7         0
secu_8         0
secu_9       560
dtype: object


In [648]:
print(df_final0521.columns.tolist())

['Num_Acc', 'place', 'catu', 'grav', 'sexe', 'trajet', 'locp', 'actp', 'etatp', 'an_nais', 'year', 'mois', 'jour', 'hrmn', 'lum', 'agg', 'int', 'atm', 'col', 'lat', 'long', 'catr', 'circ', 'nbv', 'vosp', 'prof', 'plan', 'lartpc', 'larrout', 'surf', 'infra', 'situ', 'vma', 'secu_-1', 'secu_0', 'secu_1', 'secu_2', 'secu_3', 'secu_4', 'secu_5', 'secu_6', 'secu_7', 'secu_8', 'secu_9']


Vérification

In [687]:
# On vérifie la moyenne de remplissage de secu_1 (ceinture) par année
# Si ça affiche des pourcentages cohérents (ex: 0.70 ou 0.80) partout, c'est bon
print("Taux de présence de la ceinture par année :")
print(df_final0521.groupby('year')['secu_1'].mean())

Taux de présence de la ceinture par année :
year
2005    0.556477
2006    0.542422
2007    0.542469
2008    0.540308
2009    0.548077
2010    0.556358
2011     0.55351
2012    0.558906
2013    0.555605
2014    0.551934
2015    0.586541
2016    0.598769
2017    0.563501
2018    0.595887
2019         0.0
2020         0.0
2021         0.0
Name: secu_1, dtype: object


#### re-vérification des colonnes des deux dataframes

Maintenant, revérifions que les dataframes ancien et récent ont bien les mêmes colonnes.

In [563]:
cols_recentes = set(df_final2224.columns)
cols_anciennes = set(df_final0521.columns)

unique_recent = cols_recentes-cols_anciennes # Soustraire les sets permet de ne garder que les colonnes uniquement présentes en 2022-2024
unique_ancien = cols_anciennes-cols_recentes # Colonnes uniquement présentes en 2005-2021

print(unique_ancien,unique_recent)

set() set()


Maintenant que les deux dataframes ont les même colonnes, faisons d'autres vérifications avant de les concaténer. 

In [564]:
#Réorganisons les colonnes par ordre alphabétique pour faciliter la comparaison
df_final0521 = df_final0521.reindex(sorted(df_final0521.columns), axis=1)
df_final2224 = df_final2224.reindex(sorted(df_final2224.columns), axis=1)

In [565]:
print(df_final0521.columns.tolist())
print(df_final2224.columns.tolist())

['Num_Acc', 'actp', 'agg', 'an_nais', 'atm', 'catr', 'catu', 'circ', 'col', 'etatp', 'grav', 'hrmn', 'infra', 'int', 'jour', 'larrout', 'lartpc', 'lat', 'locp', 'long', 'lum', 'mois', 'nbv', 'place', 'plan', 'prof', 'secu_-1', 'secu_0', 'secu_1', 'secu_2', 'secu_3', 'secu_4', 'secu_5', 'secu_6', 'secu_7', 'secu_8', 'secu_9', 'sexe', 'situ', 'surf', 'trajet', 'vma', 'vosp', 'year']
['Num_Acc', 'actp', 'agg', 'an_nais', 'atm', 'catr', 'catu', 'circ', 'col', 'etatp', 'grav', 'hrmn', 'infra', 'int', 'jour', 'larrout', 'lartpc', 'lat', 'locp', 'long', 'lum', 'mois', 'nbv', 'place', 'plan', 'prof', 'secu_-1', 'secu_0', 'secu_1', 'secu_2', 'secu_3', 'secu_4', 'secu_5', 'secu_6', 'secu_7', 'secu_8', 'secu_9', 'sexe', 'situ', 'surf', 'trajet', 'vma', 'vosp', 'year']


#### Regardons le type des données dans les deux dataframes, et harmonisons les

In [567]:
# Regardons les différents types
for col in df_final0521.columns:
     print(col, df_final0521[col].dtype, df_final2224[col].dtype)


Num_Acc float64 int64
actp object object
agg float64 int64
an_nais float64 float64
atm float64 int64
catr float64 int64
catu int64 int64
circ float64 int64
col float64 int64
etatp float64 int64
grav int64 int64
hrmn object object
infra float64 int64
int float64 int64
jour float64 int64
larrout float64 object
lartpc float64 object
lat object object
locp float64 int64
long object object
lum float64 int64
mois float64 int64
nbv float64 object
place float64 int64
plan float64 int64
prof float64 int64
secu_-1 int64 bool
secu_0 bool bool
secu_1 bool bool
secu_2 bool bool
secu_3 bool bool
secu_4 bool bool
secu_5 int64 bool
secu_6 int64 bool
secu_7 int64 bool
secu_8 int64 bool
secu_9 bool bool
sexe int64 int64
situ float64 int64
surf float64 int64
trajet float64 int64
vma float64 int64
vosp float64 int64
year int64 int64


Traitons rapidement la varaible "actp" pour pouvoir la convertir en Int64

In [204]:
df_final2224.loc[df_final2224['actp'] == 'A', 'actp'] = 9
df_final2224.loc[df_final2224['actp'] == 'B', 'actp'] = -1

In [ ]:
# Colonnes entières classiques
int_cols = ['Num_Acc', 'actp', 'agg', 'atm', 'col', 'etatp', 'int', 'jour', 'locp', 'lum', 'mois', 'place', 'trajet']
float_col = ['catr', 'circ', 'infra', 'larrout', 'lartpc', 'nbv', 'plan', 'prof', 'situ', 'surf', 'vma', 'vosp']
# Colonnes sécurité (toutes celles qui commencent par secu_)
secu_cols = [c for c in df_final0521.columns if c.startswith("secu_")]
# Colonnes à convertir = toutes les colonnes codées
all_int_cols = int_cols + secu_cols

for c in all_int_cols:
    # On remplace d'abord les NaN par pandas.NA pour être compatible Int64
    df_final0521[c] = df_final0521[c].where(df_final0521[c].notna(), pd.NA)
    df_final2224[c] = df_final2224[c].where(df_final2224[c].notna(), pd.NA)
    
    # Conversion en Int64 nullable
    df_final0521[c] = df_final0521[c].astype("Int64")
    df_final2224[c] = df_final2224[c].astype("Int64")

for c in float_col:
    # Remplacer les virgules par des points et convertir en float, les erreurs deviennent NaN
    df_final0521[c] = pd.to_numeric(df_final0521[c].astype(str).str.replace(',', '.'), errors='coerce')
    df_final2224[c] = pd.to_numeric(df_final2224[c].astype(str).str.replace(',', '.'), errors='coerce')

    # Maintenant les deux datasets ont float64 pour cette colonne
    df_final0521[c] = df_final0521[c].astype("float64")
    df_final2224[c] = df_final2224[c].astype("float64")




In [183]:
# Vérification finale des types
for col in df_final0521.columns:
    if df_final0521[col].dtype != df_final2224[col].dtype:
        print(col, df_final0521[col].dtype, df_final2224[col].dtype)


C'est bon, les colonnes sont identiques et dans le même ordre, et les valeurs sont de même type, on peut donc concaténer.

In [184]:
df_final = pd.concat([df_final0521, df_final2224], ignore_index=True)


In [185]:
df_final 

,Num_Acc,actp,adr,agg,an_nais,atm,catr,catu,circ,col,...,secu_7,secu_8,secu_9,sexe,situ,surf,trajet,vma,vosp,year
0,200500000001,0,CD41B,2,1976.0,1,3.0,1,2.0,3,...,0,0,0,1,1.0,1.0,1,NaN,0.0,2005
1,200500000001,0,CD41B,2,1968.0,1,3.0,1,2.0,3,...,0,0,0,2,1.0,1.0,3,NaN,0.0,2005
2,200500000001,0,CD41B,2,1964.0,1,3.0,2,2.0,3,...,0,0,0,1,1.0,1.0,0,NaN,0.0,2005
3,200500000001,0,CD41B,2,2004.0,1,3.0,2,2.0,3,...,0,0,0,1,1.0,1.0,0,NaN,0.0,2005
4,200500000001,0,CD41B,2,1998.0,1,3.0,2,2.0,3,...,0,0,0,1,1.0,1.0,0,NaN,0.0,2005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2887253,202400054401,-1,JEAN JAURES (BOULEVARD) 63/215 - 70/208,2,1978.0,1,3.0,1,2.0,3,...,0,0,0,2,1.0,1.0,0,50.0,0.0,2024
2887254,202400054401,-1,JEAN JAURES (BOULEVARD) 63/215 - 70/208,2,1984.0,1,3.0,1,2.0,3,...,0,0,0,1,1.0,1.0,0,50.0,0.0,2024
2887255,202400054402,-1,Avenue de la Pompadour,2,1981.0,1,3.0,1,1.0,4,...,0,0,0,1,1.0,1.0,4,50.0,0.0,2024
2887256,202400054402,-1,Avenue de la Pompadour,2,1986.0,1,3.0,1,1.0,4,...,0,0,0,2,1.0,1.0,9,50.0,0.0,2024


In [186]:
print(df_final.columns)


Index(['Num_Acc', 'actp', 'adr', 'agg', 'an_nais', 'atm', 'catr', 'catu',
       'circ', 'col', 'com', 'dep', 'etatp', 'grav', 'hrmn', 'infra', 'int',
       'jour', 'larrout', 'lartpc', 'lat', 'locp', 'long', 'lum', 'mois',
       'nbv', 'num_veh', 'place', 'plan', 'prof', 'secu_-1', 'secu_0',
       'secu_1', 'secu_2', 'secu_3', 'secu_4', 'secu_5', 'secu_6', 'secu_7',
       'secu_8', 'secu_9', 'sexe', 'situ', 'surf', 'trajet', 'vma', 'vosp',
       'year'],
      dtype='object')


## 5. One-hot encoding et abandonner les variables qui ne sont pas pertinentes


On peut drop "num_veh" ainsi que "adr".

In [187]:
df_final.drop(columns=["num_veh"],inplace=True)

In [188]:
df_final.drop(columns=["adr"],inplace=True)

In [189]:
print(df_final.columns)


Index(['Num_Acc', 'actp', 'agg', 'an_nais', 'atm', 'catr', 'catu', 'circ',
       'col', 'com', 'dep', 'etatp', 'grav', 'hrmn', 'infra', 'int', 'jour',
       'larrout', 'lartpc', 'lat', 'locp', 'long', 'lum', 'mois', 'nbv',
       'place', 'plan', 'prof', 'secu_-1', 'secu_0', 'secu_1', 'secu_2',
       'secu_3', 'secu_4', 'secu_5', 'secu_6', 'secu_7', 'secu_8', 'secu_9',
       'sexe', 'situ', 'surf', 'trajet', 'vma', 'vosp', 'year'],
      dtype='object')


One-hot encoding.

#### atm

In [191]:
# On regarde les lignes où atm est vide OU 0, et on affiche l'année
vides_par_an = df_final[df_final['atm'].isna() | (df_final['atm'] == 0)]['year'].value_counts()
print("Répartition des 'vides' par année :")
print(vides_par_an.sort_index())

Répartition des 'vides' par année :
year
2009        13
2010         3
2011        31
2012         9
2013        31
2014        29
2017        24
2018         9
2019     90130
2020    105295
2021    114426
Name: count, dtype: int64


On voit bien que les cellules non renseignées datent de la période 2005-2021, quand le "-1" pour "Non renseigné" n'avait pas encore été introduit. On les remplace donc par un -1.

In [192]:
df_final.loc[~df_final['atm'].isin(range(1, 10)), 'atm'] = -1

In [193]:
col_final_atm = ["atm_Non_renseigné","atm_Normale", "atm_pluie_légère", "atm_pluie_forte", "atm_neie_grêle", "atm_brouillard_fumée", "atm_vent_fort_tempête", "atm_temps_eblouissant", "atm_temps_couvert", "atm_autre" ]

atm_dummies = pd.get_dummies(df_final['atm'])

mapping = {
    -1: "atm_Non_renseigné",
     1: "atm_Normale",
     2: "atm_pluie_légère",
     3: "atm_pluie_forte",
     4: "atm_neige_grêle",
     5: "atm_brouillard_fumée",
     6: "atm_vent_fort_tempête",
     7: "atm_temps_eblouissant",
     8: "atm_temps_couvert",
     9: "atm_autre"
}

atm_dummies = atm_dummies.rename(columns=mapping)

df_final = pd.concat([df_final, atm_dummies], axis=1)

df_final.drop(columns=["atm"],inplace=True)

#### actp

Dans la base de données 2022-2024 apparaissent "-1 - Non renseigné", "A - Monte / Descend du véhicule", "B - Inconnue". 
Nous devons donc remplacer tous les vide ou NaN datant de la période 2005-2021 par des -1, nous allons aussi remplacer tous les B par des -1, et enfin nous allons remplacer tous les A par des 9 qui correspondent à "autre".  Nous remplaçons aussi "0 - Non renseigné ou sans objet" par "-1", si le 0 n'est pas trop significatif.
Puis nous ferrons le one-hot encoding.

In [198]:
ratio = (df_final['actp'].isin([0, '0']).sum()) / df_final['actp'].count()
print (ratio)

0.8087778137315441


In [201]:
nb_zeros = (df_final['actp'] == 0).sum()
print(nb_zeros)

1907822


In [ ]:
df_final.loc[df_final['actp'] == 'A', 'actp'] = 9
df_final.loc[df_final['actp'] == 'B', 'actp'] = -1
df_final.loc[df_final['actp'] == '0', 'actp'] = -1

In [ ]:
df_final.loc[df_final['actp'] == -1, 'actp'] = 0


In [61]:
mapping_actp = {
    -1: "actp_Non_renseigné",
     0: "actp_Sans_objet",
     1: "actp_Sens_vehicule",
     2: "actp_Sens_inverse",
     3: "actp_Traversant",
     4: "actp_Masqué",
     5: "actp_Jouant_courant",
     6: "actp_Avec_animal",
     9: "actp_Autre"
}

actp_dummies = pd.get_dummies(df_final['actp'])

actp_dummies = actp_dummies.reindex(mapping_actp.keys(), axis=1, fill_value=0)

actp_dummies = actp_dummies.rename(columns=mapping_actp)

df_final = pd.concat([df_final, actp_dummies], axis=1)

df_final.drop(columns='actp', inplace=True)


In [62]:
df_final.loc[df_final['catr'] == 7, 'catr'] = 9


In [63]:
mapping_catr = {
    1: "catr_Autoroute",
    2: "catr_Route_nationale",
    3: "catr_Route_departementale",
    4: "catr_Voie_communale",
    5: "catr_Hors_reseau_public",
    6: "catr_Parc_stationnement_public",
    9: "catr_Autre"
}
catr_dummies = pd.get_dummies(df_final['catr'])
catr_dummies = catr_dummies.reindex(mapping_catr.keys(), axis=1, fill_value=0)
catr_dummies = catr_dummies.rename(columns=mapping_catr)
df_final = pd.concat([df_final, catr_dummies], axis=1)
df_final.drop(columns='catr', inplace=True)


In [64]:
#Pour catu je garde le 4 de la base de donnée récente.
mapping_catu = {
    1: "catu_Conducteur",
    2: "catu_Passager",
    3: "catu_Pieton",
    4: "catu_Pieton_roller_trottinette"
}
catu_dummies = pd.get_dummies(df_final['catu'])
catu_dummies = catu_dummies.reindex(mapping_catu.keys(), axis=1, fill_value=0)
catu_dummies = catu_dummies.rename(columns=mapping_catu)
df_final = pd.concat([df_final, catu_dummies], axis=1)
df_final.drop(columns='catu', inplace=True)


In [65]:
df_final.loc[~df_final['circ'].isin(range(1, 5)), 'circ'] = -1


In [66]:
mapping_circ = {
    -1: "circ_Non_renseigné",
    1: "circ_sens_unique",
    2: "circ_bidirectionnel",
    3: "circ_chaussee_sepraree",
    4: "circ_voies-d_affectation_varaible"
}
circ_dummies = pd.get_dummies(df_final['circ'])
circ_dummies = circ_dummies.reindex(mapping_circ.keys(), axis=1, fill_value=0)
circ_dummies = circ_dummies.rename(columns=mapping_circ)
df_final = pd.concat([df_final, circ_dummies], axis=1)
df_final.drop(columns='circ', inplace=True)

In [67]:
df_final.loc[~df_final['col'].isin(range(1, 7)), 'col'] = -1


In [68]:
mapping_col = {
    -1: "col_Non_renseigné",
    1: "col_deux_vehicules_frontal",
    2: "col_deux_vehicules_arrière",
    3: "col_deux_vehicules_coté",
    4: "col_trois_vehicules_en_chaines",
    5 : "col_trois_vehicules_collisions_multiples",
    6: "col_autre_collision",
    7: "col_sans_collision"
}
col_dummies = pd.get_dummies(df_final['col'])
col_dummies = col_dummies.reindex(mapping_col.keys(), axis=1, fill_value=0)
col_dummies = col_dummies.rename(columns=mapping_col)
df_final = pd.concat([df_final, col_dummies], axis=1)
df_final.drop(columns='col', inplace=True)

In [69]:
df_final.loc[~df_final['etatp'].isin(range(1, 4)), 'col'] = -1


In [70]:
mapping_etatp = {
    -1: "etatp_non_renseigné",
    1: "etatp_seul",
    2: "etatp_accompagné",
    3: "etatp_en_groupe",
}
etatp_dummies = pd.get_dummies(df_final['etatp'])
etatp_dummies = etatp_dummies.reindex(mapping_etatp.keys(), axis=1, fill_value=0)
etatp_dummies = etatp_dummies.rename(columns=mapping_etatp)
df_final = pd.concat([df_final, etatp_dummies], axis=1)
df_final.drop(columns='etatp', inplace=True)

In [71]:
df_final.loc[~df_final['infra'].isin(range(1, 8)), 'infra'] = -1
df_final.loc[df_final['infra'] == 7, 'infra'] = 9



In [72]:
mapping_infra = {
    -1: "infra_non_renseigné",
    0: "infra_aucun",
    1: "infra_souterrain_tunnel",
    2: "infra_pont_autopont",
    3: "infra_bretelle_d_echangeur_raccordement",
    4: "infra__voie_ferrée",
    5: "infra_carrefour_aménagé",
    6: "infra_zone_pietonne",
    7: "infra_zone_péage",
    8: "infra_chantier",
    9: "infra_autres"
}
infra_dummies = pd.get_dummies(df_final['infra'])
infra_dummies = infra_dummies.reindex(mapping_infra.keys(), axis=1, fill_value=0)
infra_dummies = infra_dummies.rename(columns=mapping_infra)
df_final = pd.concat([df_final, infra_dummies], axis=1)
df_final.drop(columns='infra', inplace=True)

#attention : 0,8 et 9 ne sont pas dans l'ancienne base - vérifer l'impact

In [73]:
df_final.drop(columns='locp', inplace=True)
df_final.drop(columns='place', inplace=True)
#ça ne me semble pas important de garder la localisation du piéton dans le cadre de notre projet

In [74]:
df_final.drop(columns='place', inplace=True)
#ça ne me semble pas important de garder la place de l'usager dans le véhicule dans le cadre de notre projet

KeyError: "['place'] not found in axis"

In [ ]:
df_final.loc[~df_final['plan'].isin(range(1, 5)), 'atm'] = -1


In [ ]:
mapping_plan = {
    -1: "plan_non_renseigné",
    1: "plan_partie rectiligne",
    2: "plan_courbe_gauche",
    3: "plan_courbe_droite",
    4: "plan_en_S"
}
plan_dummies = pd.get_dummies(df_final['plan'])
plan_dummies = plan_dummies.reindex(mapping_etatp.keys(), axis=1, fill_value=0)
plan_dummies = plan_dummies.rename(columns=mapping_etatp)
df_final = pd.concat([df_final, plan_dummies], axis=1)
df_final.drop(columns='plan', inplace=True)

In [ ]:
# j'ai fait le one-hot encoding jusqu'à plan 

Pour rappel, on a : 
 
- Num_Acc: numéro d'identifiant de l'accident
- actp : action du piéton
- agg : si l'accident a eu lieu dans une agglomération
- an_nais : année de naissance de l'usager
- atm : conditions atmosphériques
- catr : catégorie de route
- catu : catégorie d'usager
- circ : régime de circulation
- col : type de collision
- com : commune 
- dep : département
- etatp : précise si le piéton accidenté était seul ou non
- grav : gravité de la blessure de l'usager
- hrmn : heure et minute de l'accident
- infra : aménagement - infrastructure
- int : intersection
- jour : jour de l'accident
- larrout : largeur de la chaussée affectée à la circulation des véhicules
- lartpc : largeur du terre plein central, s'il existe
- lat : latitude
- locp : localisation du piéton
- long : longitude
- lum : conditions d'éclairage
- mois : mois de l'accident
- nbv : nombre total de voies de circulation
- place : permet de situer la place occupée dans le véhicule par l'usager au moment de l'accident
- plan : tracé en plan
- prof : profil en long décrit la déclivité de la route à l'endroit de l'accident
- les données de sécurité
- sexe : sexe de l'usager
- situ : situation de l'accident
- surf : état de la surface
- trajet : motif de déplacement de l'accident
- vma : vitesse maximale autorisée
- vosp : signal l'existence d'une voie réservée, indépendemment du fait que l'accident ait lieu sur cette voie
- year : année de l'accident 

reovir pour dep et com

In [ ]:
df_final

,Num_Acc,actp,agg,an_nais,atm,catr,catu,circ,col,com,...,secu_7,secu_8,secu_9,sexe,situ,surf,trajet,vma,vosp,year
0,200500000001,0,2,1976.0,1,3.0,1,2.0,3,11,...,0,0,0,1,1.0,1.0,1,NaN,0.0,2005
1,200500000001,0,2,1968.0,1,3.0,1,2.0,3,11,...,0,0,0,2,1.0,1.0,3,NaN,0.0,2005
2,200500000001,0,2,1964.0,1,3.0,2,2.0,3,11,...,0,0,0,1,1.0,1.0,0,NaN,0.0,2005
3,200500000001,0,2,2004.0,1,3.0,2,2.0,3,11,...,0,0,0,1,1.0,1.0,0,NaN,0.0,2005
4,200500000001,0,2,1998.0,1,3.0,2,2.0,3,11,...,0,0,0,1,1.0,1.0,0,NaN,0.0,2005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2887253,202400054401,-1,2,1978.0,1,3.0,1,2.0,3,92012,...,0,0,0,2,1.0,1.0,0,50.0,0.0,2024
2887254,202400054401,-1,2,1984.0,1,3.0,1,2.0,3,92012,...,0,0,0,1,1.0,1.0,0,50.0,0.0,2024
2887255,202400054402,-1,2,1981.0,1,3.0,1,1.0,4,94028,...,0,0,0,1,1.0,1.0,4,50.0,0.0,2024
2887256,202400054402,-1,2,1986.0,1,3.0,1,1.0,4,94028,...,0,0,0,2,1.0,1.0,9,50.0,0.0,2024


In [ ]:
print(df_final.columns)

Index(['Num_Acc', 'agg', 'an_nais', 'com', 'dep', 'grav', 'hrmn', 'int',
       'jour', 'larrout', 'lartpc', 'lat', 'long', 'lum', 'mois', 'nbv',
       'prof', 'secu_-1', 'secu_0', 'secu_1', 'secu_2', 'secu_3', 'secu_4',
       'secu_5', 'secu_6', 'secu_7', 'secu_8', 'secu_9', 'sexe', 'situ',
       'surf', 'trajet', 'vma', 'vosp', 'year', 'atm_Non_renseigné',
       'atm_Normale', 'atm_pluie_légère', 'atm_pluie_forte', 'atm_neige_grêle',
       'atm_brouillard_fumée', 'atm_vent_fort_tempête',
       'atm_temps_eblouissant', 'atm_temps_couvert', 'atm_autre',
       'actp_Non_renseigné', 'actp_Sans_objet', 'actp_Sens_vehicule',
       'actp_Sens_inverse', 'actp_Traversant', 'actp_Masqué',
       'actp_Jouant_courant', 'actp_Avec_animal', 'actp_Autre',
       'catr_Autoroute', 'catr_Route_nationale', 'catr_Route_departementale',
       'catr_Voie_communale', 'catr_Hors_reseau_public',
       'catr_Parc_stationnement_public', 'catr_Autre', 'catu_Conducteur',
       'catu_Passager', 'catu